# Notebook 3: Noise-Conditional Score Network (NCSN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebooks/03_ncsn.ipynb)

**Goal:** Train a noise-conditioned score network $s_\theta(x, \sigma)$ that estimates $\nabla_x \log p_\sigma(x)$ across a geometric sequence of noise levels. Sample using annealed Langevin dynamics and compare with the analytical score. Introduce the modified Langevin dynamics framework.

**Key idea:** Instead of a single $\sigma$, we train on $L$ noise levels $\sigma_1 > \sigma_2 > \cdots > \sigma_L$ and condition the network on $\sigma$. Sampling proceeds from $\sigma_1$ (coarse) down to $\sigma_L$ (fine), giving the particles a "curriculum" that first finds the modes and then sharpens them.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if os.path.exists("modified-langevin-score-matching"):
        !cd modified-langevin-score-matching && git pull
    else:
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, "..")

In [ ]:
import torch
import matplotlib.pyplot as plt

from dsm import (
    make_dataset, make_dataloader,
    ScoreNetwork,
    GeometricNoiseSchedule,
    annealed_langevin_dynamics,
    modified_langevin_dynamics,
    train,
    make_8gaussians_analytical_score,
    save_checkpoint,
)
from dsm.visualization import plot_samples, plot_training_curves, animate_sampling, display_animation

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dataset
DATASET = "8gaussians"
N_SAMPLES = 10_000
BATCH_SIZE = 512

# Noise schedule
SIGMA_MIN = 0.01
SIGMA_MAX = 1.0
NUM_NOISE_LEVELS = 10

# Model architecture
HIDDEN_DIM = 256
NUM_RES_BLOCKS = 3

# Training
N_EPOCHS = 200
LR = 1e-3
SIGMA_WEIGHTING = False

# Sampling
N_GENERATED = 2000
STEPS_PER_SIGMA = 100
STEP_SIZE_FACTOR = 5e-5
SAVE_EVERY = 5  # total steps = 10 * 100 = 1000, so ~200 frames

## Create Dataset

In [ ]:
dataset = make_dataset(DATASET, n_samples=N_SAMPLES)
data = dataset.tensors[0]
dataloader = make_dataloader(dataset, batch_size=BATCH_SIZE)
print(f"Dataset shape: {data.shape}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=1, alpha=0.5)
ax.set_title(f"{DATASET} dataset ({N_SAMPLES} points)")
ax.set_aspect("equal")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
plt.show()

## Create Model & Noise Schedule

In [ ]:
model = ScoreNetwork(data_dim=2, hidden_dim=HIDDEN_DIM, num_res_blocks=NUM_RES_BLOCKS)
noise_schedule = GeometricNoiseSchedule(
    sigma_min=SIGMA_MIN, sigma_max=SIGMA_MAX, num_levels=NUM_NOISE_LEVELS
)

n_params = sum(p.numel() for p in model.parameters())
print(f"ScoreNetwork: {n_params:,} parameters")
print(f"Noise levels (descending): {noise_schedule.sigmas.tolist()}")
print(model)

## Train with Multi-Level DSM Loss

In [ ]:
history = train(
    model,
    dataloader,
    noise_schedule=noise_schedule,
    n_epochs=N_EPOCHS,
    lr=LR,
    device=DEVICE,
    log_every=40,
    sigma_weighting=SIGMA_WEIGHTING,
)

In [ ]:
plot_training_curves(history)
plt.show()

## Save Checkpoint

In [ ]:
CHECKPOINT_DIR = "checkpoints" if "google.colab" not in sys.modules else "modified-langevin-score-matching/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

save_checkpoint(
    model,
    os.path.join(CHECKPOINT_DIR, "8gaussians_trained.pt"),
    noise_schedule=noise_schedule,
    config={
        "dataset": DATASET,
        "hidden_dim": HIDDEN_DIM,
        "num_res_blocks": NUM_RES_BLOCKS,
        "n_epochs": N_EPOCHS,
        "sigma_min": SIGMA_MIN,
        "sigma_max": SIGMA_MAX,
        "num_noise_levels": NUM_NOISE_LEVELS,
    },
    history=history,
)

## Annealed Langevin Sampling with Learned Score

In [ ]:
learned_samples, learned_traj = annealed_langevin_dynamics(
    model,
    noise_schedule,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Learned samples: {learned_samples.shape}, trajectory frames: {learned_traj.shape[0]}")

plot_samples(data, learned_samples.cpu(), title="NCSN Annealed Langevin Sampling")
plt.show()

## Annealed Langevin Sampling with Analytical Score

In [ ]:
analytical_model = make_8gaussians_analytical_score().to(DEVICE)

analytical_samples, analytical_traj = annealed_langevin_dynamics(
    analytical_model,
    noise_schedule,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Analytical samples: {analytical_samples.shape}")

plot_samples(data, analytical_samples.cpu(), title="Analytical Annealed Langevin Sampling")
plt.show()

## 3-Panel Comparison: Real | Analytical | Learned

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

data_np = data.numpy()
learned_np = learned_samples.cpu().numpy()
analytical_np = analytical_samples.cpu().numpy()

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

axes[0].scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.5)
axes[0].set_title("Real Data")

axes[1].scatter(analytical_np[:, 0], analytical_np[:, 1], s=1, alpha=0.5, color="C2")
axes[1].set_title("Analytical Score")

axes[2].scatter(learned_np[:, 0], learned_np[:, 1], s=1, alpha=0.5, color="C1")
axes[2].set_title("Learned NCSN")

fig.suptitle("Annealed Langevin: Analytical vs Learned Score", fontsize=14)
fig.tight_layout()
plt.show()

## Animation: Learned NCSN Sampling

In [ ]:
anim_learned = animate_sampling(
    learned_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="NCSN Annealed Langevin",
    trail_length=10,
)
display_animation(anim_learned)

## Animation: Analytical Score Sampling

In [ ]:
anim_analytical = animate_sampling(
    analytical_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Analytical Annealed Langevin",
    trail_length=10,
)
display_animation(anim_analytical)

## Modified Langevin Dynamics

Standard annealed Langevin uses the score of the noised distribution $\nabla_x \log p_\sigma(x)$, but ideally we want to sample from the clean distribution $p_0(x)$. The **modified Langevin** framework adds a correction term to account for this gap:

$$x_{t+1} = x_t + \frac{\alpha}{2} s_\theta(x_t, \sigma) + \text{correction}(x_t, s_\theta, \sigma, \alpha) + \sqrt{\alpha}\, z_t$$

When `correction_fn=None`, modified Langevin is identical to standard annealed Langevin.

In [ ]:
# Standard annealed (for comparison)
std_learned_samples = annealed_langevin_dynamics(
    model,
    noise_schedule,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=False,
)

# Modified with no correction (should be identical to standard annealed)
mod_learned_samples = modified_langevin_dynamics(
    model,
    noise_schedule,
    correction_fn=None,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=False,
)

# Analytical with modified (no correction)
mod_analytical_samples = modified_langevin_dynamics(
    analytical_model,
    noise_schedule,
    correction_fn=None,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=False,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

mod_analytical_np = mod_analytical_samples.cpu().numpy()
std_learned_np = std_learned_samples.cpu().numpy()
mod_learned_np = mod_learned_samples.cpu().numpy()

axes[0].scatter(mod_analytical_np[:, 0], mod_analytical_np[:, 1], s=1, alpha=0.5, color="C2")
axes[0].set_title("Analytical (Standard Annealed)")

axes[1].scatter(std_learned_np[:, 0], std_learned_np[:, 1], s=1, alpha=0.5, color="C1")
axes[1].set_title("Learned (Standard Annealed)")

axes[2].scatter(mod_learned_np[:, 0], mod_learned_np[:, 1], s=1, alpha=0.5, color="C3")
axes[2].set_title("Learned (Modified, no correction)")

fig.suptitle("Standard vs Modified Langevin (correction_fn=None)", fontsize=14)
fig.tight_layout()
plt.show()

As expected, with `correction_fn=None` the modified sampler produces the same distribution as standard annealed Langevin (up to random seed differences). The correction term is defined and explored in Notebook 04.

## Placeholder Correction Function

Define your correction term below and re-run the modified sampling cells above to see its effect.

In [ ]:
def correction_fn(x, score, sigma, alpha):
    """Correction term for modified Langevin dynamics.
    
    Args:
        x: (n_samples, data_dim) current particle positions
        score: (n_samples, data_dim) score estimate s(x, sigma)
        sigma: scalar, current noise level
        alpha: scalar, current step size
    
    Returns:
        correction: (n_samples, data_dim) additive correction
    """
    # TODO: Your correction term here
    return torch.zeros_like(x)

In [ ]:
# Re-run modified sampling with the correction function
mod_corrected_samples, mod_corrected_traj = modified_langevin_dynamics(
    model,
    noise_schedule,
    correction_fn=correction_fn,
    n_samples=N_GENERATED,
    data_dim=2,
    steps_per_sigma=STEPS_PER_SIGMA,
    step_size_factor=STEP_SIZE_FACTOR,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

axes[0].scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.5)
axes[0].set_title("Real Data")

axes[1].scatter(std_learned_np[:, 0], std_learned_np[:, 1], s=1, alpha=0.5, color="C1")
axes[1].set_title("Standard Annealed")

mod_corrected_np = mod_corrected_samples.cpu().numpy()
axes[2].scatter(mod_corrected_np[:, 0], mod_corrected_np[:, 1], s=1, alpha=0.5, color="C3")
axes[2].set_title("Modified (with correction)")

fig.suptitle("Effect of Correction Term", fontsize=14)
fig.tight_layout()
plt.show()

## Summary

- The noise-conditional ScoreNetwork (~414k params) learns accurate scores across multiple noise levels simultaneously.
- **Annealed Langevin dynamics** leverages this multi-scale structure: coarse noise levels navigate globally, fine levels refine locally.
- The analytical score provides an upper bound on achievable quality -- any gap is due to score estimation error.
- The **modified Langevin** framework is ready for a correction term. With `correction_fn=None`, it reduces to standard annealed Langevin.

**Next:** In Notebook 04, we explore the correction term in detail and compare standard vs modified Langevin quantitatively.